# Book Market Analytics: Books to Scrape
**Data Science 500 Assignment** | Richfield Graduate Institute of Technology

**Author:** *<your name and student ITS number>*

## Project aim
A publishing analytics team wants to understand which factors influence book prices and ratings on
[Books to Scrape](https://books.toscrape.com). This notebook documents the full pipeline:

1. Data acquisition (web scraping with BeautifulSoup)
2. Data cleaning and preparation
3. Data visualisation and exploration
4. Predictive modelling (Linear Regression)
5. Interpretation, limitations and recommendations

**Scope decision:** I scrape the **entire catalogue (1 000 books, 50 categories)**. A larger sample gives more
stable statistics and lets me compare genres, which a single-genre scrape (often only 10 to 30 books) cannot do.

**Reproducibility:** run all cells top to bottom. The scrape takes roughly 5 to 10 minutes because of a polite
delay between requests. The raw data is cached to disk, so later runs reload it instead of re-scraping.

## 0. Setup

In [ ]:
import os, re, time, warnings
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

BASE_URL    = "https://books.toscrape.com/"
CATALOGUE   = BASE_URL + "catalogue/"
RAW_CSV     = "books_raw.csv"
CLEAN_CSV   = "books_clean.csv"
FORCE_RESCRAPE = False      # set True to ignore the cached raw file
REQUEST_DELAY  = 0.15       # seconds between requests (be polite to the server)

## 1. Data Acquisition (Web Scraping)

**Approach**
- *Stage 1 (listing pages):* loop through `catalogue/page-N.html` until the server returns 404. For each product card I
  extract the **title, price, star rating, availability text** and the link to the product page.
- *Stage 2 (product pages):* visit each product page to get fields not on the listing: **category (genre), description,
  stock quantity, UPC, number of reviews, tax**.
- A shared `requests.Session` with retries and a delay keeps the scraper robust and respectful.
- Note: the site sends UTF-8 without declaring it, so I set `response.encoding = "utf-8"`. Otherwise the pound sign
  becomes `Â£`.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (educational scraping project, Richfield DS500)"})

def get_soup(url, retries=3):
    """Download a page and return BeautifulSoup, or None if the page does not exist (404)."""
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=15)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            r.encoding = "utf-8"                     # fixes the 'Â£' encoding problem
            time.sleep(REQUEST_DELAY)
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"  retry {attempt+1}/{retries} for {url}: {e}")
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Failed to fetch {url}")

def parse_listing_page(soup, page_url):
    """Extract the required fields from every product card on one listing page."""
    rows = []
    for card in soup.select("article.product_pod"):
        link = card.h3.a
        rows.append({
            "title":        link["title"],                                   # full title (visible text is truncated)
            "price_raw":    card.select_one("p.price_color").get_text(strip=True),
            "rating_word":  card.select_one("p.star-rating")["class"][1],    # e.g. 'Three'
            "availability_raw": card.select_one("p.instock.availability").get_text(strip=True),
            "product_url":  urljoin(page_url, link["href"]),
        })
    return rows

def scrape_listings():
    all_rows, page = [], 1
    while True:
        url = f"{CATALOGUE}page-{page}.html"
        soup = get_soup(url)
        if soup is None:
            break
        all_rows.extend(parse_listing_page(soup, url))
        print(f"listing page {page:>2}: total {len(all_rows)} books", end="\r")
        page += 1
    print()
    return pd.DataFrame(all_rows)

In [ ]:
def parse_product_page(url):
    """Extract genre, description and product-information table from one product page."""
    soup = get_soup(url)
    crumbs = [li.get_text(strip=True) for li in soup.select("ul.breadcrumb li")]
    category = crumbs[2] if len(crumbs) >= 3 else None            # Home > Books > <Category> > Title

    desc_tag = soup.select_one("#product_description")            # header; the text is in the next <p>
    description = None
    if desc_tag is not None:
        p = desc_tag.find_next_sibling("p")
        description = p.get_text(" ", strip=True) if p else None

    info = {}
    for tr in soup.select("table.table-striped tr"):
        info[tr.th.get_text(strip=True)] = tr.td.get_text(strip=True)

    return {
        "category":        category,
        "description":     description,
        "upc":             info.get("UPC"),
        "price_excl_raw":  info.get("Price (excl. tax)"),
        "price_incl_raw":  info.get("Price (incl. tax)"),
        "tax_raw":         info.get("Tax"),
        "num_reviews_raw": info.get("Number of reviews"),
        "availability_detail": info.get("Availability"),
    }

def scrape_catalogue():
    listing = scrape_listings()
    print(f"Collected {len(listing)} listings. Scraping product pages ...")
    details = []
    for i, url in enumerate(listing["product_url"], 1):
        details.append(parse_product_page(url))
        if i % 50 == 0 or i == len(listing):
            print(f"  product pages: {i}/{len(listing)}", end="\r")
    print()
    return pd.concat([listing, pd.DataFrame(details)], axis=1)

if os.path.exists(RAW_CSV) and not FORCE_RESCRAPE:
    raw = pd.read_csv(RAW_CSV)
    print(f"Loaded cached raw data: {raw.shape}")
else:
    raw = scrape_catalogue()
    raw.to_csv(RAW_CSV, index=False)
    print(f"Scraped and saved raw data: {raw.shape}")

raw.head()

In [ ]:
print("Rows:", len(raw), "| Columns:", list(raw.columns))
print("\nRaw examples of the messy fields:")
print(raw[["price_raw", "rating_word", "availability_raw", "availability_detail"]].head(5).to_string())
assert len(raw) >= 200, "Assignment requires at least 200 books"

## 2. Data Cleaning and Preparation

**Issues expected in the raw data and the planned treatment**

| Issue | Treatment and justification |
|---|---|
| Price stored as text with currency symbol (`£51.77`) | Strip symbol, convert to `float` |
| Rating stored as a word (`Three`) | Map to integer 1 to 5 (it is ordinal, so an integer keeps the order) |
| Availability stored as text (`In stock (22 available)`) | Split into a boolean `in_stock` and an integer `stock_qty` |
| Missing descriptions | Keep the book. Fill with an empty string and add a flag `has_description`, so no rows are lost |
| Descriptions contain HTML entities / stray whitespace / odd characters | Normalise whitespace and fix encoding artefacts |
| Possible duplicates | Drop by UPC (unique product code) |
| Category names inconsistent in case or spacing | Strip and title-case |
| Constant columns (e.g. number of reviews) | Check and document. Exclude from modelling if there is no variance |

In [ ]:
df = raw.copy()

# 1. Duplicates
before = len(df)
df = df.drop_duplicates(subset="upc").reset_index(drop=True)
print(f"Duplicates removed: {before - len(df)}")

# 2. Prices -> float (GBP)
def to_price(s):
    return float(re.sub(r"[^0-9.]", "", str(s))) if pd.notna(s) else np.nan
df["price"]      = df["price_raw"].apply(to_price)
df["price_excl"] = df["price_excl_raw"].apply(to_price)
df["price_incl"] = df["price_incl_raw"].apply(to_price)
df["tax"]        = df["tax_raw"].apply(to_price)

# 3. Rating word -> integer
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating_word"].map(rating_map)

# 4. Availability -> in_stock flag + quantity
df["in_stock"] = df["availability_raw"].str.contains("In stock", case=False, na=False)
df["stock_qty"] = (df["availability_detail"].fillna(df["availability_raw"])
                   .str.extract(r"(\d+)")[0].astype(float))
df.loc[~df["in_stock"], "stock_qty"] = 0
df["stock_qty"] = df["stock_qty"].fillna(0).astype(int)

# 5. Reviews
df["num_reviews"] = pd.to_numeric(df["num_reviews_raw"], errors="coerce").fillna(0).astype(int)

# 6. Text fields
def clean_text(s):
    if pd.isna(s):
        return ""
    s = s.replace("Â", "").replace("â\x80\x99", "'").replace("â\x80\x9c", '"').replace("â\x80\x9d", '"')
    s = re.sub(r"\s+", " ", s)
    return s.strip()

df["title"]       = df["title"].apply(clean_text)
df["description"] = df["description"].apply(clean_text)
df["category"]    = df["category"].fillna("Unknown").str.strip().str.title()
df["has_description"] = (df["description"] != "").astype(int)

# 7. Derived text features (prepared for analysis)
df["title_len"]       = df["title"].str.len()
df["title_words"]     = df["title"].str.split().str.len()
df["desc_char_len"]   = df["description"].str.len()
df["desc_word_count"] = df["description"].str.split().str.len().fillna(0).astype(int)

# 8. Final tidy dataset
clean = df[["upc", "title", "category", "price", "rating", "rating_word", "in_stock", "stock_qty",
            "num_reviews", "has_description", "title_len", "title_words",
            "desc_char_len", "desc_word_count", "description", "product_url"]].copy()
clean.to_csv(CLEAN_CSV, index=False)
print("Clean dataset:", clean.shape, "-> saved to", CLEAN_CSV)
clean.head()

In [ ]:
print("Missing values per column:")
print(clean.isna().sum()[clean.isna().sum() > 0] if clean.isna().sum().sum() else "None")
print("\nBooks with no description:", (clean["has_description"] == 0).sum())

print("\nData types:")
print(clean.dtypes.to_string())

print("\nValidity checks")
print("Prices in range:", clean["price"].between(5, 100).all(), f"(min £{clean.price.min()}, max £{clean.price.max()})")
print("Ratings in 1-5:", clean["rating"].between(1, 5).all())
print("Listing price matches product-page price incl. tax:", np.allclose(clean["price"], df["price_incl"].fillna(clean["price"])))

print("\nColumns with no variation (cannot help a model):")
for c in ["in_stock", "num_reviews"]:
    print(f"  {c}: {clean[c].nunique()} unique value(s) -> {clean[c].unique()[:3]}")

clean.describe().round(2).T

**Cleaning decisions and findings**
- *(Fill in after running)* Note how many duplicates and missing descriptions were found.
- `in_stock` and `num_reviews` usually have **no variation** on this sandbox site (all books are in stock, all have
  0 reviews). They are retained in the dataset for completeness but **excluded from the model**, since a constant
  column carries no information.
- The site has **no user "tags"**. Genre (`category`) is the closest equivalent, and the free text is the description,
  from which I derive length features and keyword counts.

## 3. Data Visualisation and Exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(clean["price"], bins=30, kde=True, ax=axes[0], color="#1f4e9e")
axes[0].axvline(clean["price"].mean(), color="red", ls="--", label=f"Mean £{clean['price'].mean():.2f}")
axes[0].axvline(clean["price"].median(), color="green", ls="--", label=f"Median £{clean['price'].median():.2f}")
axes[0].set(title="Distribution of book prices", xlabel="Price (GBP)", ylabel="Number of books"); axes[0].legend()

sns.boxplot(x=clean["price"], ax=axes[1], color="#8fb4ea")
axes[1].set(title="Price spread and outliers", xlabel="Price (GBP)")
plt.tight_layout(); plt.show()
print(clean["price"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = ["One", "Two", "Three", "Four", "Five"]
sns.countplot(x="rating_word", data=clean, order=order, ax=axes[0], color="#1f4e9e")
axes[0].set(title="Number of books per star rating", xlabel="Star rating", ylabel="Books")

sns.boxplot(x="rating_word", y="price", data=clean, order=order, ax=axes[1], color="#8fb4ea")
axes[1].set(title="Price by star rating", xlabel="Star rating", ylabel="Price (GBP)")
plt.tight_layout(); plt.show()

print(clean.groupby("rating")["price"].agg(["count", "mean", "median", "std"]).round(2))

In [ ]:
cat_stats = (clean.groupby("category")
             .agg(books=("price", "size"), avg_price=("price", "mean"),
                  avg_rating=("rating", "mean"), avg_stock=("stock_qty", "mean"))
             .round(2).sort_values("books", ascending=False))

top = cat_stats.head(15).index
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(y="category", x="price", data=clean[clean["category"].isin(top)],
            order=cat_stats.loc[top].sort_values("avg_price").index, color="#8fb4ea", ax=ax)
ax.set(title="Price distribution in the 15 largest categories", xlabel="Price (GBP)", ylabel="")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cat_stats[cat_stats.books >= 10].avg_price.sort_values().tail(12).plot.barh(ax=axes[0], color="#1f4e9e")
axes[0].set(title="Highest average price (categories with 10+ books)", xlabel="Average price (GBP)", ylabel="")
cat_stats[cat_stats.books >= 10].avg_rating.sort_values().tail(12).plot.barh(ax=axes[1], color="#2e8b57")
axes[1].set(title="Highest average rating (categories with 10+ books)", xlabel="Average star rating", ylabel="")
plt.tight_layout(); plt.show()
cat_stats.head(15)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.scatterplot(x="desc_word_count", y="price", data=clean, alpha=.5, ax=axes[0])
sns.regplot(x="desc_word_count", y="price", data=clean, scatter=False, color="red", ax=axes[0])
axes[0].set(title="Price vs description length", xlabel="Description words", ylabel="Price (GBP)")

sns.scatterplot(x="stock_qty", y="price", data=clean, alpha=.5, ax=axes[1])
sns.regplot(x="stock_qty", y="price", data=clean, scatter=False, color="red", ax=axes[1])
axes[1].set(title="Price vs units in stock", xlabel="Units in stock", ylabel="Price (GBP)")

sns.scatterplot(x="title_len", y="price", data=clean, alpha=.5, ax=axes[2])
sns.regplot(x="title_len", y="price", data=clean, scatter=False, color="red", ax=axes[2])
axes[2].set(title="Price vs title length", xlabel="Title characters", ylabel="Price (GBP)")
plt.tight_layout(); plt.show()

In [ ]:
num_cols = ["price", "rating", "stock_qty", "desc_word_count", "desc_char_len", "title_len", "title_words"]
corr = clean[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix of numeric variables")
plt.tight_layout(); plt.show()
print("Correlation of each variable with price:")
print(corr["price"].drop("price").sort_values(key=abs, ascending=False).round(3))

In [ ]:
# Text exploration: most common description keywords, and do 'expensive' books use different words?
stop = set("""the a an and or but of to in on for with is are was were be been it its as at by from this that these those
he she they his her their i you we our your who whom which what when where how not no so if than then there about into
out up down over more most one two will can has have had also all any just very only after before while like new
life story book books""".split())

def tokens(text):
    return [w for w in re.findall(r"[a-z]{3,}", text.lower()) if w not in stop]

median_price = clean["price"].median()
hi = pd.Series([w for d in clean.loc[clean.price > median_price, "description"] for w in tokens(d)]).value_counts()
lo = pd.Series([w for d in clean.loc[clean.price <= median_price, "description"] for w in tokens(d)]).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
hi.head(12)[::-1].plot.barh(ax=axes[0], color="#c0392b"); axes[0].set_title(f"Top words: books above median (£{median_price:.2f})")
lo.head(12)[::-1].plot.barh(ax=axes[1], color="#1f4e9e"); axes[1].set_title("Top words: books at or below median")
plt.tight_layout(); plt.show()

### Interpretation of visual findings
*(Rewrite in your own words after you have looked at your own plots. Suggested prompts below.)*

1. **Price distribution:** Is it symmetric or skewed? What are the mean, median and range? Roughly what share of books
   sit in the £10 to £60 band?
2. **Ratings:** Are ratings spread evenly or concentrated? Does price differ visibly across star levels? (Compare the
   medians in the box plot and the table.)
3. **Genres:** Which categories are largest? Which are priciest or best rated? Are those averages reliable given the
   category sizes (many categories have fewer than 10 books)?
4. **Relationships:** Are any correlations with price above about |0.1|? A near-zero correlation heatmap is itself
   an insight: observable listing features explain little of the price on this site.
5. **Text:** Do description length or keywords relate to price?

## 4. Predictive Modelling: Linear Regression

**Target:** `price` (GBP).

**Candidate features and justification**

| Feature | Type | Why include it |
|---|---|---|
| `rating` (1 to 5) | numeric/ordinal | Quality signal that could plausibly affect price |
| `category` | categorical | Genres may follow different pricing conventions |
| `stock_qty` | numeric | Inventory level may relate to price (e.g. discounting overstock) |
| `desc_word_count` | numeric | Proxy for how much marketing content a title gets |
| `title_len` | numeric | Simple text-derived feature |
| `has_description` | binary | Controls for missing descriptions |

**Excluded:** `in_stock` and `num_reviews` (constant, so no information). `price_excl`, `price_incl` and `tax` (these
are the target in disguise, so using them would be **data leakage**). `upc`, `title`, `product_url` (identifiers).

**Handling many categories:** categories with fewer than 10 books are grouped as "Other" so one-hot columns are not
dominated by tiny groups. The pipeline fits encoding and scaling on the training data only, which avoids leakage.

In [ ]:
model_df = clean.copy()
counts = model_df["category"].value_counts()
rare = counts[counts < 10].index
model_df["category_grp"] = model_df["category"].where(~model_df["category"].isin(rare), "Other")
print(f"{model_df.category_grp.nunique()} category groups after merging {len(rare)} rare categories into 'Other'")

num_features = ["rating", "stock_qty", "desc_word_count", "title_len", "has_description"]
cat_features = ["category_grp"]
X = model_df[num_features + cat_features]
y = model_df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_features),
])

lin_model = Pipeline([("prep", preprocess), ("lr", LinearRegression())])
baseline  = Pipeline([("prep", preprocess), ("dummy", DummyRegressor(strategy="mean"))])

lin_model.fit(X_train, y_train)
baseline.fit(X_train, y_train)

def evaluate(model, Xa, ya, name):
    p = model.predict(Xa)
    return {"model": name, "MAE": mean_absolute_error(ya, p),
            "RMSE": np.sqrt(mean_squared_error(ya, p)), "R2": r2_score(ya, p)}

results = pd.DataFrame([
    evaluate(baseline,  X_train, y_train, "Baseline (mean) - train"),
    evaluate(baseline,  X_test,  y_test,  "Baseline (mean) - test"),
    evaluate(lin_model, X_train, y_train, "Linear Regression - train"),
    evaluate(lin_model, X_test,  y_test,  "Linear Regression - test"),
]).round(3)
print(results.to_string(index=False))

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2  = cross_val_score(lin_model, X, y, cv=cv, scoring="r2")
cv_mae = -cross_val_score(lin_model, X, y, cv=cv, scoring="neg_mean_absolute_error")
print(f"\n5-fold CV R2 : {cv_r2.mean():.3f} (+/- {cv_r2.std():.3f})")
print(f"5-fold CV MAE: £{cv_mae.mean():.2f} (+/- {cv_mae.std():.2f})")

In [ ]:
pred = lin_model.predict(X_test)
resid = y_test - pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
axes[0].scatter(y_test, pred, alpha=.6)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--"); axes[0].set(title="Actual vs predicted price", xlabel="Actual (GBP)", ylabel="Predicted (GBP)")

axes[1].scatter(pred, resid, alpha=.6); axes[1].axhline(0, color="r", ls="--")
axes[1].set(title="Residuals vs predicted", xlabel="Predicted (GBP)", ylabel="Residual (GBP)")

sns.histplot(resid, kde=True, ax=axes[2], color="#1f4e9e"); axes[2].set(title="Residual distribution", xlabel="Residual (GBP)")
plt.tight_layout(); plt.show()

In [ ]:
feature_names = lin_model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lin_model.named_steps["lr"].coef_, index=feature_names).sort_values(key=abs, ascending=False)
print(f"Intercept: £{lin_model.named_steps['lr'].intercept_:.2f}")
print("\nLargest coefficients (numeric ones are per 1 standard deviation; categories are vs. the reference category):")
print(coefs.head(12).round(3).to_string())

coefs.head(12)[::-1].plot.barh(figsize=(8, 4.5), color="#1f4e9e")
plt.title("Largest linear-regression coefficients (GBP)"); plt.tight_layout(); plt.show()

In [ ]:
# Statistical significance with statsmodels (OLS on the full data, same features)
Xs = pd.get_dummies(model_df[num_features + cat_features], columns=cat_features, drop_first=True).astype(float)
ols = sm.OLS(y, sm.add_constant(Xs)).fit()
print(ols.summary().tables[0])
sig = ols.pvalues.drop("const").sort_values()
print("\nFeatures with p < 0.05:", list(sig[sig < 0.05].index) or "none")
print("Overall F-test p-value:", round(ols.f_pvalue, 4))

In [ ]:
# Linear regression assumptions: multicollinearity (VIF) for numeric features
from statsmodels.stats.outliers_influence import variance_inflation_factor
Xv = sm.add_constant(model_df[num_features].astype(float))
vif = pd.Series([variance_inflation_factor(Xv.values, i) for i in range(1, Xv.shape[1])], index=num_features)
print("Variance inflation factors (values above ~5 suggest multicollinearity):")
print(vif.round(2).to_string())

### Interpretation and limitations
*(Complete with your own numbers.)*

- **Performance:** Compare the linear model's test R², MAE and RMSE to the baseline that always predicts the mean
  price. If R² is close to 0 (or negative on the test set), the features carry little predictive signal.
- **Drivers:** Which coefficients are largest, and are they statistically significant (statsmodels p-values)?
  Remember that the scaled numeric coefficients are in £ per 1 standard deviation.
- **Overfitting check:** Compare train and test scores, and the cross-validation spread.
- **Limitations to discuss:**
  1. Books to Scrape is a **sandbox site**: prices appear to be assigned largely at random, so weak predictability is
     expected and does not mean the method is wrong.
  2. Only listing-level features are available (no author, publisher, page count, publication year or sales).
  3. Many categories are small, so category effects are noisy.
  4. Linear regression assumes a linear, additive relationship and roughly constant error variance. Check the residual plots.
  5. Results describe **association, not causation**, and may not generalise to real retailers.
- **Next steps:** try regularised models (Ridge/Lasso), tree-based models, TF-IDF text features, and richer real
  retailer data.

## 5. Summary of Key Results (auto-generated)

In [ ]:
test_row = results[results["model"] == "Linear Regression - test"].iloc[0]
base_row = results[results["model"] == "Baseline (mean) - test"].iloc[0]
top_cat = cat_stats[cat_stats.books >= 10].avg_price.idxmax()
low_cat = cat_stats[cat_stats.books >= 10].avg_price.idxmin()

print(f"Books analysed ................ {len(clean)} across {clean.category.nunique()} categories")
print(f"Average price ................. £{clean.price.mean():.2f} (median £{clean.price.median():.2f}, range £{clean.price.min():.2f} to £{clean.price.max():.2f})")
print(f"Most common rating ............ {clean.rating_word.mode()[0]} stars")
print(f"Priciest category (10+ books) . {top_cat} (£{cat_stats.loc[top_cat, 'avg_price']:.2f})")
print(f"Cheapest category (10+ books) . {low_cat} (£{cat_stats.loc[low_cat, 'avg_price']:.2f})")
print(f"Correlation rating-price ...... {clean[['rating','price']].corr().iloc[0,1]:.3f}")
print(f"Model test R2 / MAE / RMSE .... {test_row.R2:.3f} / £{test_row.MAE:.2f} / £{test_row.RMSE:.2f}")
print(f"Baseline test R2 / MAE ........ {base_row.R2:.3f} / £{base_row.MAE:.2f}")
print(f"5-fold CV R2 .................. {cv_r2.mean():.3f}")